In [54]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

In [55]:
df = pd.read_csv("../data/dataset.csv")

feature_cols = [c for c in df.columns if c not in ['datetime', 'underlying_price']]
ce_cols = [c for c in feature_cols if c.endswith('CE')]
pe_cols = [c for c in feature_cols if c.endswith('PE')]

print(f"Shape: {df.shape}")
print(f"CE columns: {len(ce_cols)}, PE columns: {len(pe_cols)}")
print(f"Total missing: {df[feature_cols].isna().sum().sum()} ({df[feature_cols].isna().sum().sum() / (df.shape[0]*len(feature_cols))*100:.1f}%)")

Shape: (975, 30)
CE columns: 14, PE columns: 14
Total missing: 5460 (20.0%)


In [56]:
def get_strike(col):
    # e.g. NIFTY27JAN2625200CE → 25200
    return int(col[12:17])

ce_strikes = np.array([get_strike(c) for c in ce_cols])
pe_strikes = np.array([get_strike(c) for c in pe_cols])

print("CE strikes:", ce_strikes)
print("PE strikes:", pe_strikes)

CE strikes: [25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]
PE strikes: [23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]


In [57]:
# Leave-one-out CV on rows with 13+ observed strikes
# Proves cross-sectional interpolation is valid with no lookahead bias

errors_cs   = []
errors_mean = []

for idx in df.index:
    row = df.iloc[idx]
    for strikes, cols in [(ce_strikes, ce_cols), (pe_strikes, pe_cols)]:
        ivs = np.array([row[c] for c in cols], dtype=float)
        obs_mask = ~np.isnan(ivs)
        if obs_mask.sum() < 13:
            continue

        obs_idx   = np.where(obs_mask)[0]
        type_mean = ivs[obs_mask].mean()

        for leave_i in obs_idx:
            train_idx = obs_idx[obs_idx != leave_i]
            if len(train_idx) < 2:
                continue
            f = interp1d(strikes[train_idx], ivs[train_idx],
                         kind='linear', fill_value='extrapolate')
            pred = float(f(strikes[leave_i]))
            errors_cs.append((ivs[leave_i] - pred) ** 2)
            errors_mean.append((ivs[leave_i] - type_mean) ** 2)

print(f"LOO test points : {len(errors_cs)}")
print(f"Cross-sectional MSE : {np.mean(errors_cs):.8f}")
print(f"Mean-fill baseline  : {np.mean(errors_mean):.8f}")
print(f"Improvement         : {np.mean(errors_mean)/np.mean(errors_cs):.0f}x better")

LOO test points : 4911
Cross-sectional MSE : 0.00004980
Mean-fill baseline  : 0.00956600
Improvement         : 192x better


In [58]:
def fill_cross_sectional(df_input, strikes, cols):
    """
    For each timestamp, fit a linear interpolant across observed strikes
    and predict all missing strikes of the same option type.
    No future timestamps are used — purely cross-sectional at each row.
    """
    df_out = df_input.copy()
    for idx in df_input.index:
        row  = df_input.iloc[idx]
        ivs  = np.array([row[c] for c in cols], dtype=float)
        obs  = ~np.isnan(ivs)
        if obs.sum() < 2:
            continue   # temporal fallback handles this
        f = interp1d(strikes[obs], ivs[obs],
                     kind='linear', fill_value='extrapolate')
        for i, col in enumerate(cols):
            if np.isnan(ivs[i]):
                df_out.at[idx, col] = float(np.clip(f(strikes[i]), 0.01, 10.0))
    return df_out

In [59]:
df_filled = df.copy()
df_filled = fill_cross_sectional(df_filled, ce_strikes, ce_cols)
df_filled = fill_cross_sectional(df_filled, pe_strikes, pe_cols)
print(f"After cross-sectional pass — still missing: {df_filled[feature_cols].isna().sum().sum()}")

# Temporal fallback for any row with < 2 observations (rare edge case)
for col in feature_cols:
    df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='forward') # <--- CORRECTED

# Final safety net
for col in feature_cols:
    if df_filled[col].isna().any():
        df_filled[col].fillna(df_filled[col].median(), inplace=True)

print(f"Final missing: {df_filled[feature_cols].isna().sum().sum()}")
print("Done!")

After cross-sectional pass — still missing: 0
Final missing: 0
Done!


In [60]:
df_filled.to_csv("../data/filled_dataset.csv", index=False)
print("Saved → data/filled_dataset.csv")

Saved → data/filled_dataset.csv


In [64]:
ORIGINAL_DATASET_PATH = "../data/dataset.csv"
SEPARATOR = "||"

def generate_solution(filled_path, output_path="../submissions/submission.csv"):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)
    feat     = [c for c in original.columns if c != "datetime"]

    rows = []
    for col in feat:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    sol = pd.DataFrame(rows, columns=["id", "value"])
    sol = sol.sort_values("id").reset_index(drop=True)
    sol.to_csv(output_path, index=False)
    print(f"✅ Submission saved → {output_path}  ({len(sol)} rows)")
    return sol

sol = generate_solution("../data/filled_dataset.csv")
print(sol.head(10))

✅ Submission saved → ../submissions/submission.csv  (5460 rows)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163440
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.113730
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.101500
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.170055
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.159770
5  07-01-2026 09:20||NIFTY27JAN2624800PE  0.128225
6  07-01-2026 09:20||NIFTY27JAN2625000PE  0.118580
7  07-01-2026 09:20||NIFTY27JAN2625300CE  0.096810
8  07-01-2026 09:20||NIFTY27JAN2625400CE  0.107300
9  07-01-2026 09:20||NIFTY27JAN2625800CE  0.106310
